# S09 — Convolution

**Week 5 · Wed Sep 23, 2026 · Module 2**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s09_convolution.ipynb)

Every cell below is a worked example from the [S09 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s09/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s09.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s09.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
try:
    import torch  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
    import torch  # noqa: F401
print("environment ready")

## The convolution operation, precisely


*Expected output starts with:* `stride=1 padding=0: output 6x6, max |numpy - torch| = 7.15e-07`


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

np.random.seed(0)
torch.manual_seed(0)

def conv2d_np(x, k, stride=1, padding=0):
    """2D 'convolution' as deep learning frameworks define it (cross-correlation).
    x: (H, W) input, k: (kh, kw) kernel."""
    if padding > 0:
        x = np.pad(x, padding)
    kh, kw = k.shape
    H, W = x.shape
    out_h = (H - kh) // stride + 1
    out_w = (W - kw) // stride + 1
    out = np.zeros((out_h, out_w), dtype=np.float32)
    for i in range(out_h):
        for j in range(out_w):
            patch = x[i * stride : i * stride + kh, j * stride : j * stride + kw]
            out[i, j] = np.sum(patch * k)
    return out

x = np.random.randn(8, 8).astype(np.float32)
k = np.random.randn(3, 3).astype(np.float32)

for stride, padding in [(1, 0), (1, 1), (2, 1)]:
    ours = conv2d_np(x, k, stride=stride, padding=padding)
    torch_out = F.conv2d(
        torch.from_numpy(x).view(1, 1, 8, 8),
        torch.from_numpy(k).view(1, 1, 3, 3),
        stride=stride, padding=padding,
    )[0, 0].numpy()
    max_diff = np.abs(ours - torch_out).max()
    print(f"stride={stride} padding={padding}: "
          f"output {ours.shape[0]}x{ours.shape[1]}, "
          f"max |numpy - torch| = {max_diff:.2e}")

## Kernels as pattern detectors


*Expected output starts with:* `edge response (one row shown; all rows identical):`


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

np.random.seed(0)
torch.manual_seed(0)

# An image that is dark (0) on the left half, bright (1) on the right half.
img = torch.zeros(1, 1, 6, 8)
img[:, :, :, 4:] = 1.0

# A hand-designed vertical-edge kernel: responds where left != right.
edge_k = torch.tensor([[-1.0, 0.0, 1.0],
                       [-1.0, 0.0, 1.0],
                       [-1.0, 0.0, 1.0]]).view(1, 1, 3, 3)

edges = F.conv2d(img, edge_k)  # no padding: 6x8 -> 4x6
print("edge response (one row shown; all rows identical):")
print(edges[0, 0, 0].tolist())

# The same kernel applied to the image shifted right by one pixel:
img_shift = torch.zeros(1, 1, 6, 8)
img_shift[:, :, :, 5:] = 1.0
edges_shift = F.conv2d(img_shift, edge_k)
print("edge response on shifted image:")
print(edges_shift[0, 0, 0].tolist())

# Max pooling makes the detection tolerant to that shift.
pooled = F.max_pool2d(edges, kernel_size=2, stride=2)
pooled_shift = F.max_pool2d(edges_shift, kernel_size=2, stride=2)
print(f"after 2x2 max pool:         {pooled[0, 0, 0].tolist()}")
print(f"after 2x2 max pool (shift): {pooled_shift[0, 0, 0].tolist()}")

# Parameter counts: conv layer vs fully connected layer on a 32x32 RGB image.
conv = torch.nn.Conv2d(3, 16, kernel_size=3, padding=1)   # 32x32x3 -> 32x32x16
fc = torch.nn.Linear(32 * 32 * 3, 32 * 32 * 16)           # same input/output sizes
n_conv = sum(p.numel() for p in conv.parameters())
n_fc = sum(p.numel() for p in fc.parameters())
print(f"conv 3->16, 3x3 kernel:  {n_conv:,} parameters")
print(f"fully connected version: {n_fc:,} parameters")
print(f"ratio: {n_fc / n_conv:,.0f}x")

## Receptive fields: how a 3x3 kernel sees the whole image


*Expected output starts with:* `layer          k  s   rf  jump`


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# Receptive-field arithmetic. Track two numbers layer by layer:
#   jump j  = product of all strides so far (input pixels per output step)
#   rf      = receptive field; each layer adds (kernel - 1) * incoming jump
layers = [
    ("conv 3x3 s1", 3, 1),
    ("conv 3x3 s1", 3, 1),
    ("avgpool 2 s2", 2, 2),
    ("conv 3x3 s1", 3, 1),
    ("avgpool 2 s2", 2, 2),
    ("conv 3x3 s1", 3, 1),
]
rf, j = 1, 1
print(f"{'layer':<13} {'k':>2} {'s':>2} {'rf':>4} {'jump':>5}")
for name, k, s in layers:
    rf = rf + (k - 1) * j
    j = j * s
    print(f"{name:<13} {k:>2} {s:>2} {rf:>4} {j:>5}")

# Empirical check: which input pixels can influence one output unit?
# Build the same stack with all-positive weights (so no cancellation),
# backprop from a single central output unit, and measure the support.
convs = []
for name, k, s in layers:
    if name.startswith("conv"):
        c = nn.Conv2d(1, 1, 3, padding=0, bias=False)
        with torch.no_grad():
            c.weight.fill_(0.1)
        convs.append(c)
    else:
        convs.append(nn.AvgPool2d(2))
net = nn.Sequential(*convs)

x = torch.ones(1, 1, 64, 64, requires_grad=True)
out = net(x)
h = out.shape[-1] // 2
out[0, 0, h, h].backward()
support = (x.grad[0, 0] != 0)
rows = support.any(dim=1).nonzero().squeeze(1)
cols = support.any(dim=0).nonzero().squeeze(1)
print(f"output map is {out.shape[-2]}x{out.shape[-1]}; "
      f"one central unit's input support: "
      f"{rows.max() - rows.min() + 1} x {cols.max() - cols.min() + 1} pixels")

## Under the hood: convolution is a matrix multiply


*Expected output starts with:* `patch matrix (27, 36), kernel matrix (16, 27)`


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

np.random.seed(0)
torch.manual_seed(0)

def im2col(x, kh, kw):
    """x: (C, H, W) -> matrix of patches (C*kh*kw, out_h*out_w). Stride 1, no padding."""
    C, H, W = x.shape
    out_h, out_w = H - kh + 1, W - kw + 1
    cols = np.zeros((C * kh * kw, out_h * out_w), dtype=x.dtype)
    col = 0
    for i in range(out_h):
        for j in range(out_w):
            cols[:, col] = x[:, i:i + kh, j:j + kw].ravel()
            col += 1
    return cols

x = np.random.randn(3, 8, 8).astype(np.float32)       # 3-channel 8x8 image
w = np.random.randn(16, 3, 3, 3).astype(np.float32)   # 16 kernels, 3x3, 3 channels

cols = im2col(x, 3, 3)          # every 3x3x3 patch, flattened into a column
W_mat = w.reshape(16, -1)       # every kernel, flattened into a row
out = (W_mat @ cols).reshape(16, 6, 6)   # ONE matrix multiply = the whole conv

ref = F.conv2d(torch.from_numpy(x)[None], torch.from_numpy(w))[0].numpy()
print(f"patch matrix {cols.shape}, kernel matrix {W_mat.shape}")
print(f"conv-as-matmul output: {out.shape}, "
      f"max |matmul - F.conv2d| = {np.abs(out - ref).max():.2e}")
print(f"patch matrix holds {cols.size:,} floats vs {x.size:,} in the image "
      f"({cols.size / x.size:.1f}x duplication)")

## Variations: dilated, grouped, and depthwise convolution


*Expected output starts with:* `standard 3x3, 64->64:           36,864 params`


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

def n_params(m):
    return sum(p.numel() for p in m.parameters())

# Four ways to map 64 channels to 64 channels with 3x3 spatial extent.
standard  = nn.Conv2d(64, 64, 3, padding=1, bias=False)
grouped   = nn.Conv2d(64, 64, 3, padding=1, groups=4, bias=False)
depthwise = nn.Conv2d(64, 64, 3, padding=1, groups=64, bias=False)
pointwise = nn.Conv2d(64, 64, 1, bias=False)

print(f"standard 3x3, 64->64:           {n_params(standard):6,d} params")
print(f"grouped 3x3 (4 groups):         {n_params(grouped):6,d} params")
print(f"depthwise 3x3 (64 groups):      {n_params(depthwise):6,d} params")
print(f"depthwise separable (dw + 1x1): {n_params(depthwise) + n_params(pointwise):6,d} params")

x = torch.randn(2, 64, 16, 16)
y = pointwise(depthwise(x))
print(f"depthwise-separable output: {tuple(y.shape)} (same as standard: "
      f"{tuple(standard(x).shape)})")

# Dilation: spread the 9 kernel taps apart without adding parameters.
img = torch.zeros(1, 1, 9, 9)
img[0, 0, 4, 4] = 1.0                    # one bright pixel in the center
k = torch.ones(1, 1, 3, 3)               # 9 taps, always 9 parameters
for d in [1, 2, 3]:
    out = F.conv2d(img, k, padding=d, dilation=d)
    hit = (out[0, 0] > 0).nonzero()
    span = hit[:, 0].max().item() - hit[:, 0].min().item() + 1
    print(f"dilation {d}: responses span {span}x{span} "
          f"-> effective kernel size {(3 - 1) * d + 1}")

## Try it yourself

1. Extend `conv2d_np` to multi-channel input and multiple kernels: input `(C_in, H, W)`, weights `(C_out, C_in, kh, kw)`, output `(C_out, H_out, W_out)`. Verify against `F.conv2d` with random tensors, as in the first example. This is most of HW3's first problem.
2. Build the horizontal-edge analog of `edge_k` and apply both kernels to an image containing one vertical and one horizontal edge. Confirm each detector fires only on its own orientation.
3. Compute the receptive field of this stack by hand: 3x3 conv, 2x2 max pool (stride 2), 3x3 conv, 2x2 max pool (stride 2), 3x3 conv. Then verify empirically: feed two images through that differ in exactly one pixel and see which output positions change.
4. Time `conv2d_np` against `F.conv2d` on a 128x128 input. The gap you observe is why real frameworks lower convolution to matrix multiplication (im2col) or specialized kernels instead of Python loops.


---

Full discussion of everything above: [S09 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s09/).
